# Teste Interativo da Classe FlightGeoMapper

Notebook de validação da classe `FlightGeoMapper` (visualizações geográficas de aeroportos, rotas e atrasos).

Mapas testados:
1. Scatter geo por taxa de atraso (cor = delay, tamanho = volume)
2. Scatter geo por cluster (perfil operacional)
3. Linhas das top rotas coloridas por delay médio
4. Heatmap interativo Folium
5. Choropleth por estado (bônus)

In [1]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto para permitir importar a classe do módulo notebooks
sys.path.append(str(Path(".").resolve().parent))

from notebooks.mapa_geografico import FlightGeoMapper

In [2]:
# Inicializa o mapper.
# Se você salvou o KMeans/scaler do notebook 04, passe os caminhos em kmeans_path e scaler_path.
# Sem isso, a classe faz um KMeans local de fallback com k=4.
mapper = FlightGeoMapper(
    input_path="../data/processed/flights_model.parquet",
    # kmeans_path="../models/kmeans_aeroportos.pkl",
    # scaler_path="../models/scaler_aeroportos.pkl",
    min_voos=500,
    n_clusters_fallback=4,
    top_n_routes=150,
)

In [3]:
# 1. Carregar coordenadas embutidas dos aeroportos
coords = mapper.load_coords()
coords.head(3)

Aeroportos com coordenadas: 181


,iata,name,lat,lon,state
0,ATL,Hartsfield-Jackson Atlanta,33.6367,-84.4281,GA
1,LAX,Los Angeles International,33.9425,-118.4081,CA
2,ORD,Chicago O'Hare,41.9742,-87.9073,IL


In [4]:
# 2. Carregar parquet processado
mapper.load_data()

Carregando dados de: ../data/processed/flights_model.parquet ...
Existe? True
Shape: (5714008, 34)


YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,is_delayed,scheduled_departure_hour,periodo_dia
i64,i64,i64,i64,str,i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i8,i64,str
2015,1,1,4,"""AS""",98,"""N407AS""","""ANC""","""SEA""",5,2354,-11,21,15,205,194,169,1448,404,4,430,408,-22,0,0,null,null,null,null,null,null,0,0,"""madrugada"""
2015,1,1,4,"""AA""",2336,"""N3KUAA""","""LAX""","""PBI""",10,2,-8,12,14,280,279,263,2330,737,4,750,741,-9,0,0,null,null,null,null,null,null,0,0,"""madrugada"""
2015,1,1,4,"""US""",840,"""N171US""","""SFO""","""CLT""",20,18,-2,16,34,286,293,266,2296,800,11,806,811,5,0,0,null,null,null,null,null,null,1,0,"""madrugada"""
2015,1,1,4,"""AA""",258,"""N3HYAA""","""LAX""","""MIA""",20,15,-5,15,30,285,281,258,2342,748,8,805,756,-9,0,0,null,null,null,null,null,null,0,0,"""madrugada"""
2015,1,1,4,"""AS""",135,"""N527AS""","""SEA""","""ANC""",25,24,-1,11,35,235,215,199,1448,254,5,320,259,-21,0,0,null,null,null,null,null,null,0,0,"""madrugada"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015,12,31,4,"""B6""",688,"""N657JB""","""LAX""","""BOS""",2359,2355,-4,22,17,320,298,272,2611,749,4,819,753,-26,0,0,null,null,null,null,null,null,0,23,"""noite"""
2015,12,31,4,"""B6""",745,"""N828JB""","""JFK""","""PSE""",2359,2355,-4,17,12,227,215,195,1617,427,3,446,430,-16,0,0,null,null,null,null,null,null,0,23,"""noite"""
2015,12,31,4,"""B6""",1503,"""N913JB""","""JFK""","""SJU""",2359,2350,-9,17,7,221,222,197,1598,424,8,440,432,-8,0,0,null,null,null,null,null,null,0,23,"""noite"""


In [5]:
# 3. Agregar perfil por aeroporto e juntar com coordenadas
ap_map = mapper.aggregate_airports()
ap_map.head(3)

Aeroportos com coords: 176 de 403 no dataset


,ORIGIN_AIRPORT,qtd_voos,media_arrival_delay,taxa_atraso,taxa_atraso_grave,std_arrival_delay,n_destinos,iata,name,lat,lon,state
0,SEA,110178,2.975567,0.395224,0.159950,31.571458,73,SEA,Seattle-Tacoma,47.4502,-122.3088,WA
1,HNL,42946,0.796465,0.360057,0.100964,36.971350,28,HNL,Daniel K. Inouye Honolulu,21.3245,-157.9251,HI
2,HSV,4423,3.394755,0.318788,0.151707,44.956471,8,HSV,Huntsville International,34.6372,-86.7751,AL


In [6]:
# 4. Atribuir clusters (tenta carregar modelo salvo; senão faz KMeans local)
mapper.assign_clusters()

Modelo não encontrado — reclusterizando localmente...
✓ Reclusterizado com k=4
cluster
0    49
1    33
2    72
3    22
Name: count, dtype: int64


,ORIGIN_AIRPORT,qtd_voos,media_arrival_delay,taxa_atraso,taxa_atraso_grave,std_arrival_delay,n_destinos,iata,name,lat,lon,state,cluster,cluster_str,perfil
0,SEA,110178,2.975567,0.395224,0.159950,31.571458,73,SEA,Seattle-Tacoma,47.4502,-122.3088,WA,3,3,Cluster 3
1,HNL,42946,0.796465,0.360057,0.100964,36.971350,28,HNL,Daniel K. Inouye Honolulu,21.3245,-157.9251,HI,1,1,Cluster 1
2,HSV,4423,3.394755,0.318788,0.151707,44.956471,8,HSV,Huntsville International,34.6372,-86.7751,AL,2,2,Cluster 2
3,BOS,104804,4.291649,0.369595,0.196567,40.700862,62,BOS,Boston Logan,42.3656,-71.0096,MA,3,3,Cluster 3
4,LCH,1802,0.971143,0.273585,0.132075,43.647867,2,LCH,Lake Charles Regional,30.1261,-93.2233,LA,1,1,Cluster 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,LAS,131937,6.023337,0.400926,0.198633,38.597716,78,LAS,Las Vegas Harry Reid,36.0840,-115.1537,NV,3,3,Cluster 3
172,CID,6588,8.714026,0.386764,0.204159,50.258027,7,CID,The Eastern Iowa,41.8847,-91.7108,IA,0,0,Cluster 0
173,LFT,5005,7.606394,0.390010,0.191009,45.918137,4,LFT,Lafayette Regional,30.2053,-91.9876,LA,0,0,Cluster 0
174,AEX,3054,8.943353,0.385724,0.190242,54.382943,3,AEX,Alexandria International,31.3274,-92.5498,LA,0,0,Cluster 0


### (Opcional) Renomear clusters após inspecionar os centróides

Depois de rodar o notebook 04 e ver o perfil de cada cluster, você pode dar nomes descritivos pros mapas ficarem mais interpretáveis:

In [ ]:
# Exemplo de rótulos descritivos (ajuste conforme seus resultados reais)
# mapper.set_cluster_labels({
#     "0": "Hubs problemáticos",
#     "1": "Regionais tranquilos",
#     "2": "Hubs eficientes",
#     "3": "Médios mistos",
# })

## Mapa 1 — Aeroportos por taxa de atraso

In [7]:
fig1 = mapper.build_map_delay_rate()
fig1.show()

## Mapa 2 — Aeroportos por cluster

In [8]:
fig2 = mapper.build_map_clusters()
fig2.show()

## Mapa 3 — Top rotas mais movimentadas coloridas por delay

In [9]:
# Constrói o dataframe das rotas (com coordenadas de origem e destino)
routes = mapper.build_routes_dataframe()
routes.head(3)

Rotas com coords completas: 139


,ORIGIN_AIRPORT,DESTINATION_AIRPORT,qtd_voos,media_delay,iata_x,lat_orig,lon_orig,iata_y,lat_dest,lon_dest
0,SFO,LAX,13400,11.440000,SFO,37.6213,-122.3790,LAX,33.9425,-118.4081
1,LAX,SFO,13109,10.739187,LAX,33.9425,-118.4081,SFO,37.6213,-122.3790
2,JFK,LAX,11853,-2.671560,JFK,40.6413,-73.7781,LAX,33.9425,-118.4081


In [10]:
fig3 = mapper.build_map_routes()
fig3.show()

## Mapa 4 — Heatmap interativo (Folium)

O mapa também é salvo em arquivo HTML para visualização fora do Jupyter.

In [11]:
# Salva o HTML no diretório atual e retorna o objeto Folium pra exibição inline
m = mapper.build_map_heatmap(save_path="mapa_atrasos_heatmap.html")
m  # Exibe inline

Mapa salvo em mapa_atrasos_heatmap.html


## Mapa 5 — Choropleth por estado (bônus)

In [12]:
fig5 = mapper.build_map_state_choropleth()
fig5.show()

In [13]:
# Inspecionar o agregado por estado
mapper.state_stats.sort_values("taxa_atraso", ascending=False).head(10)

,state,taxa_atraso,media_delay,n_aeroportos
18,MD,0.414792,7.248432,1
30,NV,0.400926,6.023337,1
12,IL,0.383735,6.803556,5
21,MN,0.379792,5.297249,3
4,CA,0.378461,4.899637,11
2,AR,0.376912,8.002043,3
39,SD,0.376384,7.538748,3
1,AL,0.375891,6.085873,4
22,MO,0.373984,6.048944,3
16,LA,0.373439,7.124826,7


## Alternativa: pipeline completo com `run_all`

In [ ]:
# Constrói todos os mapas de uma vez (os fig ficam acessíveis via mapper.fig_*)
# mapper_full = FlightGeoMapper(
#     input_path="../data/processed/flights_model.parquet",
# )
# mapper_full.run_all(show_plots=True)